# Part 3 — Ingestion: PDF Loader & Chunker

This notebook demonstrates the correct functioning of `src/ingestion/pdf_loader.py` (EP-02, TASK-09).

| Function | Description |
|----------|-------------|
| `load_pdf(path)` | Extracts text from each page of a PDF with PyMuPDF |
| `chunk_documents(docs)` | Splits `Document` objects into semantically coherent `Chunk` objects |
| `load_and_chunk_pdf(path)` | Convenience: load + chunk in a single call |

**Chunking strategy:** `RecursiveCharacterTextSplitter` with separators `["\n\n", "\n", ". ", " "]`, tiktoken-based token length (`cl100k_base`), `chunk_size=512`, `chunk_overlap=64`.

> The notebook creates a synthetic PDF in memory from example business glossary text (same source as project fixtures) via PyMuPDF, then loads and chunks it.

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## 1. Creating a synthetic PDF with PyMuPDF

Since the ingestion pipeline operates on PDFs, we create a test document programmatically from the same source used by fixtures: `tests/fixtures/sample_docs/business_glossary.txt`. The PDF is saved in a temporary directory.

In [2]:
import tempfile
import fitz  # pymupdf

# Read the example business glossary
glossary_path = project_root / "tests" / "fixtures" / "sample_docs" / "business_glossary.txt"
glossary_text = glossary_path.read_text(encoding="utf-8")

print(f"Business glossary: {len(glossary_text)} characters, {len(glossary_text.splitlines())} lines")
print("\nFirst 5 lines:")
for line in glossary_text.splitlines()[:5]:
    print(f"  {line}")

Business glossary: 2371 characters, 30 lines

First 5 lines:
  BUSINESS GLOSSARY — E-COMMERCE DOMAIN
  
  Customer
  A Customer is any individual or legal entity that has registered an account on the platform and has made at least one purchase. Customers are identified by a unique Customer ID and may have one or more associated delivery addresses. Synonyms: client, buyer, account holder.
  


In [3]:
import textwrap

# Split text into simulated pages (one per main concept)
# In a real PDF each page would be extracted by the governance office
sections = glossary_text.strip().split("\n\n")
sections = [s.strip() for s in sections if s.strip()]

print(f"Sections identified: {len(sections)}")
for i, s in enumerate(sections, 1):
    print(f"  Section {i}: {s[:60].replace(chr(10), ' ')}...")

Sections identified: 9
  Section 1: BUSINESS GLOSSARY — E-COMMERCE DOMAIN...
  Section 2: Customer A Customer is any individual or legal entity that h...
  Section 3: Product A Product is a tangible or digital item offered for ...
  Section 4: SalesOrder A SalesOrder is a formal transactional document t...
  Section 5: OrderLineItem An OrderLineItem is a single line within a Sal...
  Section 6: Category A Category is a hierarchical classification groupin...
  Section 7: CustomerAddress A CustomerAddress is a postal or geographic ...
  Section 8: Inventory Inventory refers to the quantity of a Product curr...
  Section 9: KEY RELATIONSHIPS: - A Customer places one or more SalesOrde...


In [4]:
# Create the synthetic PDF: one section per page
tmp_dir = Path(tempfile.mkdtemp())
pdf_path = tmp_dir / "business_glossary.pdf"

doc = fitz.open()  # new empty document
for section in sections:
    page = doc.new_page(width=595, height=842)  # A4
    # Insert text with automatic wrapping
    rect = fitz.Rect(50, 50, 545, 792)
    page.insert_textbox(
        rect,
        section,
        fontsize=11,
        fontname="helv",
        color=(0, 0, 0),
    )

doc.save(str(pdf_path))
doc.close()

file_size_kb = pdf_path.stat().st_size / 1024
print(f"PDF created: {pdf_path}")
print(f"Size: {file_size_kb:.1f} KB")
print(f"Pages: {len(sections)}")

PDF created: /tmp/tmpehrda8en/business_glossary.pdf
Size: 5.7 KB
Pages: 9


## 2. `load_pdf()` — Text extraction

`load_pdf()` opens the PDF with `fitz.open()`, iterates over each page, extracts text with `page.get_text("text")`, and creates a `Document` for each non-empty page. Handles missing, corrupt, or password-protected PDFs with `IngestionError`.

In [5]:
from src.ingestion.pdf_loader import load_pdf, IngestionError
from src.models.schemas import Document

documents = load_pdf(pdf_path)

print(f"Documents extracted: {len(documents)}")
print()
for i, doc in enumerate(documents[:4], 1):
    text_preview = doc.text[:80].replace("\n", " ")
    print(f"Document {i}:")
    print(f"  metadata: {doc.metadata}")
    print(f"  text (first 80): '{text_preview}'")
    print()

/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')
{"ts": "2026-03-12T16:39:30", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Loaded 9 pages from 'business_glossary.pdf'"}


Documents extracted: 9

Document 1:
  metadata: {'source': 'business_glossary.pdf', 'page': '1'}
  text (first 80): 'BUSINESS GLOSSARY ? E-COMMERCE DOMAIN'

Document 2:
  metadata: {'source': 'business_glossary.pdf', 'page': '2'}
  text (first 80): 'Customer A Customer is any individual or legal entity that has registered an acc'

Document 3:
  metadata: {'source': 'business_glossary.pdf', 'page': '3'}
  text (first 80): 'Product A Product is a tangible or digital item offered for sale on the platform'

Document 4:
  metadata: {'source': 'business_glossary.pdf', 'page': '4'}
  text (first 80): 'SalesOrder A SalesOrder is a formal transactional document that records the agre'



In [6]:
# Verify error handling

# 1. Non-existent file
try:
    load_pdf(Path("/tmp/nonexistent.pdf"))
except IngestionError as e:
    print(f"✅ Missing file → IngestionError: {e}")

# 2. All documents have expected metadata
for doc in documents:
    assert "source" in doc.metadata, "metadata must contain 'source'"
    assert "page" in doc.metadata, "metadata must contain 'page'"
    assert isinstance(doc, Document)

print(f"✅ All {len(documents)} documents have 'source' and 'page' metadata")
print(f"   source: '{documents[0].metadata['source']}'")
print(f"   page range: {documents[0].metadata['page']} → {documents[-1].metadata['page']}")

✅ Missing file → IngestionError: PDF file not found: /tmp/nonexistent.pdf
✅ All 9 documents have 'source' and 'page' metadata
   source: 'business_glossary.pdf'
   page range: 1 → 9


## 3. `chunk_documents()` — Semantic chunking

`chunk_documents()` uses `RecursiveCharacterTextSplitter` with tiktoken token length (`cl100k_base`). Parameters come from `settings.chunk_size` and `settings.chunk_overlap`. Each chunk inherits parent document metadata and adds `token_count`.

In [7]:
from src.ingestion.pdf_loader import chunk_documents
from src.config.settings import settings

print(f"Chunking parameters:")
print(f"  chunk_size:    {settings.chunk_size} tokens")
print(f"  chunk_overlap: {settings.chunk_overlap} tokens")
print()

chunks = chunk_documents(documents)

print(f"Chunks produced: {len(chunks)} (from {len(documents)} documents)")
if chunks:
    token_counts = [int(c.metadata.get("token_count", 0)) for c in chunks]
    print(f"Tokens per chunk: min={min(token_counts)}, max={max(token_counts)}, avg={sum(token_counts)/len(token_counts):.1f}")

{"ts": "2026-03-12T16:39:30", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Chunked 9 documents into 9 chunks (chunk_size=512, overlap=64)"}


Chunking parameters:
  chunk_size:    512 tokens
  chunk_overlap: 64 tokens

Chunks produced: 9 (from 9 documents)
Tokens per chunk: min=12, max=76, avg=53.8


In [8]:
# Detailed inspection of first chunks
print("First 5 chunks detail:")
print("=" * 70)
for chunk in chunks[:5]:
    text_preview = chunk.text[:120].replace("\n", " ")
    print(f"[chunk_index={chunk.chunk_index}]")
    print(f"  source={chunk.metadata.get('source')}, page={chunk.metadata.get('page')}, tokens={chunk.metadata.get('token_count')}")
    print(f"  text: '{text_preview}'")
    print()

First 5 chunks detail:
[chunk_index=0]
  source=business_glossary.pdf, page=1, tokens=12
  text: 'BUSINESS GLOSSARY ? E-COMMERCE DOMAIN'

[chunk_index=1]
  source=business_glossary.pdf, page=2, tokens=56
  text: 'Customer A Customer is any individual or legal entity that has registered an account on the platform and has made at lea'

[chunk_index=2]
  source=business_glossary.pdf, page=3, tokens=63
  text: 'Product A Product is a tangible or digital item offered for sale on the platform. Each Product is uniquely identified by'

[chunk_index=3]
  source=business_glossary.pdf, page=4, tokens=76
  text: 'SalesOrder A SalesOrder is a formal transactional document that records the agreement between the platform and a Custome'

[chunk_index=4]
  source=business_glossary.pdf, page=5, tokens=62
  text: 'OrderLineItem An OrderLineItem is a single line within a SalesOrder that specifies a particular Product, the quantity or'



In [9]:
# Verify all chunks
for chunk in chunks:
    assert chunk.chunk_index >= 0
    assert "source" in chunk.metadata
    assert "page" in chunk.metadata
    assert "token_count" in chunk.metadata
    assert int(chunk.metadata["token_count"]) <= settings.chunk_size, \
        f"chunk {chunk.chunk_index} exceeds chunk_size: {chunk.metadata['token_count']} > {settings.chunk_size}"

print(f"✅ All {len(chunks)} chunks:")
print(f"   · have sequential chunk_index")
print(f"   · inherit source and page metadata")
print(f"   · have token_count ≤ chunk_size ({settings.chunk_size})")

✅ All 9 chunks:
   · have sequential chunk_index
   · inherit source and page metadata
   · have token_count ≤ chunk_size (512)


## 4. `load_and_chunk_pdf()` — Convenience function

The `load_and_chunk_pdf()` function combines `load_pdf()` and `chunk_documents()` in a single call, producing chunks ready for the SLM.

In [10]:
from src.ingestion.pdf_loader import load_and_chunk_pdf

# Single call: load + chunk
all_chunks = load_and_chunk_pdf(pdf_path)

# Must produce same result as separate pipeline
assert len(all_chunks) == len(chunks), "Chunks must be identical to separate pipeline"
print(f"load_and_chunk_pdf(): {len(all_chunks)} chunks ✅")
print(f"Identical to load_pdf() + chunk_documents(): {len(chunks)} chunks")

{"ts": "2026-03-12T16:39:30", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Loaded 9 pages from 'business_glossary.pdf'"}
{"ts": "2026-03-12T16:39:30", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Chunked 9 documents into 9 chunks (chunk_size=512, overlap=64)"}


load_and_chunk_pdf(): 9 chunks ✅
Identical to load_pdf() + chunk_documents(): 9 chunks


## 5. Visualization: token distribution per chunk

In [11]:
# Token distribution without external graphic dependencies
token_counts = [int(c.metadata.get("token_count", 0)) for c in chunks]

# ASCII histogram
buckets = {"0-64": 0, "65-128": 0, "129-256": 0, "257-384": 0, "385-512": 0}
for t in token_counts:
    if t <= 64: buckets["0-64"] += 1
    elif t <= 128: buckets["65-128"] += 1
    elif t <= 256: buckets["129-256"] += 1
    elif t <= 384: buckets["257-384"] += 1
    else: buckets["385-512"] += 1

max_count = max(buckets.values()) if buckets.values() else 1
bar_width = 40

print("Token distribution per chunk (ASCII histogram):")
print()
for label, count in buckets.items():
    bar = "█" * int(count / max_count * bar_width)
    print(f"  {label:9s} | {bar:<40s} {count}")
print()
print(f"  Total chunks: {len(token_counts)}")
print(f"  Average:      {sum(token_counts)/len(token_counts):.1f} tokens")
print(f"  Min:          {min(token_counts)} tokens")
print(f"  Max:          {max(token_counts)} tokens")
print(f"  chunk_size:   {settings.chunk_size} tokens (configured limit)")

Token distribution per chunk (ASCII histogram):

  0-64      | ████████████████████████████████████████ 7
  65-128    | ███████████                              2
  129-256   |                                          0
  257-384   |                                          0
  385-512   |                                          0

  Total chunks: 9
  Average:      53.8 tokens
  Min:          12 tokens
  Max:          76 tokens
  chunk_size:   512 tokens (configured limit)


In [12]:
# Cleanup
import shutil
shutil.rmtree(tmp_dir)
print(f"Temporary directory removed: {tmp_dir}")

Temporary directory removed: /tmp/tmpehrda8en


## Summary — Part 3 Architecture (Ingestion)

```
PDF file
   │
   ▼
load_pdf(path: Path) → list[Document]
   │  · fitz.open() → page by page
   │  · page.get_text("text") + strip()
   │  · metadata: {source, page}
   │  · IngestionError on: missing / encrypted / corrupt
   │
   ▼
chunk_documents(docs) → list[Chunk]
   │  · RecursiveCharacterTextSplitter
   │  · separators: ["\n\n", "\n", ". ", " "]
   │  · length_function: tiktoken cl100k_base
   │  · chunk_size=512, chunk_overlap=64 (from settings)
   │  · metadata: {source, page, token_count}
   │  · sequential global chunk_index
   │
   ▼
list[Chunk]  →  Extract_Triplets_SLM (Part 4)
```

**TASK-10 (DDL Parser) and TASK-11 (Schema Enricher)** are not yet implemented — the respective files are empty stubs. They will be demonstrated in the complete Part 3 notebook when implemented.